# Multi-Contract Model Runs (AG, AL, AU, BB, BU)This notebook trains four models (Logistic Regression, SVM, Random Forest, XGBoost if available) on five contracts (AG, AL, AU, BB, BU) using the shared Python modules in `esl_project`. Each model has its own training cell followed by a visualization cell. Figures are saved with timestamped names into per-contract folders.

In [ ]:
from pathlib import Pathfrom datetime import datetimeimport importlib.utilimport matplotlib.pyplot as pltimport pandas as pdimport seaborn as snsfrom esl_project.data_utils import DataLoadConfig, list_csv_filesfrom esl_project.pipelines import ContractRunConfig, run_single_contract, short_contract_tagplt.style.use("seaborn-v0_8-whitegrid")RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")OUTPUT_ROOT = Path("outputs_parallel") / f"ESL_run_{RUN_TIMESTAMP}"OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)DATA_DIR = Path("2005年__20250905")CONTRACT_CODES = ["AG", "AL", "AU", "BB", "BU"]print(f"Run timestamp: {RUN_TIMESTAMP}")print(f"Output root: {OUTPUT_ROOT.resolve()}")

In [ ]:
all_files = list_csv_files(DATA_DIR)contract_paths = {}for code in CONTRACT_CODES:    matches = [p for p in all_files if p.name.startswith(f"{code}_")]    if not matches:        print(f"[Warn] No CSV found for {code} under {DATA_DIR}")        continue    contract_paths[code] = matches[0]if not contract_paths:    raise FileNotFoundError("None of the requested contract CSVs were found.")else:    for code, path in contract_paths.items():        print(f"Using {code}: {path.name}")load_cfg = DataLoadConfig(    data_dir=DATA_DIR,    max_files=None,    nrows_per_file=None,    start_date="2014-01-01",    end_date="2020-12-31",)

In [ ]:
def plot_confusion_and_rolling(model_label: str, tag: str, results: dict):    "Plot confusion heatmap and rolling metrics, then save with timestamped names."    fig_dir = results["fig_dir"]    fig_dir.mkdir(parents=True, exist_ok=True)    conf = pd.DataFrame(results["confusion"], index=[-1, 0, 1], columns=[-1, 0, 1])    metrics_df = results["rolling_metrics"]    fig, ax = plt.subplots(figsize=(5, 4))    sns.heatmap(conf, annot=True, fmt="g", cmap="Blues", ax=ax)    ax.set_title(f"{tag} {model_label} Confusion Matrix")    ax.set_xlabel("Predicted")    ax.set_ylabel("True")    fig.savefig(fig_dir / f"{tag}_{model_label}_confusion_{RUN_TIMESTAMP}.png", dpi=150, bbox_inches="tight")    plt.show()    fig, ax = plt.subplots(figsize=(7, 4))    if not metrics_df.empty:        ax.plot(metrics_df["test_end"], metrics_df["accuracy"], marker="o", label="Accuracy")        ax.plot(metrics_df["test_end"], metrics_df["f1_macro"], marker="o", label="Macro F1")        ax.tick_params(axis="x", labelrotation=45)        fig.autofmt_xdate()    ax.set_title(f"{tag} {model_label} Rolling Performance")    ax.set_xlabel("Test month end")    ax.set_ylabel("Score")    ax.legend()    fig.savefig(fig_dir / f"{tag}_{model_label}_rolling_{RUN_TIMESTAMP}.png", dpi=150, bbox_inches="tight")    plt.show()

In [ ]:
logit_cfg = ContractRunConfig(    candidate_C=[0.01, 0.1],    model_name="logit",    train_months=12,    test_months=1,    downsample_every=2,)logit_results = {}for code, path in contract_paths.items():    tag = short_contract_tag(path.stem)    res = run_single_contract(path, OUTPUT_ROOT, logit_cfg, load_cfg)    logit_results[code] = res    print(f"[Logit][{code}] Best params: {res.get('best_params')}")    print(f"[Logit][{code}] Summary head:{res.get('summary').head()}")

In [ ]:
for code, res in logit_results.items():    plot_confusion_and_rolling("logit", short_contract_tag(res.get("contract", code)), res)

In [ ]:
svm_cfg = ContractRunConfig(    candidate_C=[0.5],    model_name="svm",    param_grid=[{"C": 0.5, "tol": 1e-3, "max_iter": 1000}],    train_months=12,    test_months=1,    downsample_every=5,)svm_results = {}for code, path in contract_paths.items():    tag = short_contract_tag(path.stem)    res = run_single_contract(path, OUTPUT_ROOT, svm_cfg, load_cfg)    svm_results[code] = res    print(f"[SVM][{code}] Best params: {res.get('best_params')}")    print(f"[SVM][{code}] Summary head:{res.get('summary').head()}")

In [ ]:
for code, res in svm_results.items():    plot_confusion_and_rolling("svm", short_contract_tag(res.get("contract", code)), res)

In [ ]:
rf_cfg = ContractRunConfig(    candidate_C=[0.1],    model_name="rf",    param_grid=[{"n_estimators": 120, "max_depth": 8, "max_features": "sqrt", "n_jobs": -1}],    train_months=12,    test_months=1,    downsample_every=5,)rf_results = {}for code, path in contract_paths.items():    tag = short_contract_tag(path.stem)    res = run_single_contract(path, OUTPUT_ROOT, rf_cfg, load_cfg)    rf_results[code] = res    print(f"[RF][{code}] Best params: {res.get('best_params')}")    print(f"[RF][{code}] Summary head:{res.get('summary').head()}")

In [ ]:
for code, res in rf_results.items():    plot_confusion_and_rolling("rf", short_contract_tag(res.get("contract", code)), res)

In [ ]:
xgb_available = importlib.util.find_spec("xgboost") is not Noneif not xgb_available:    print("[XGB] xgboost is not installed; skipping this run.")    xgb_results = Noneelse:    import numpy as np    import esl_project.model_loader as ml    from esl_project.models.xgboost_model import train_xgboost as _train_xgb    _orig_train_model = ml.train_model    class XGBLabelWrapper:        def __init__(self, base_model, inv_map):            self.model = base_model            self.inv_map = inv_map            self._order = [-1, 0, 1]        def predict(self, X):            raw = self.model.predict(X)            return np.array([self.inv_map[int(v)] for v in raw])        def predict_proba(self, X):            base = self.model.predict_proba(X)            out = np.zeros((base.shape[0], len(self._order)))            mapping = {0: -1, 1: 0, 2: 1}            for src_lbl, tgt_lbl in mapping.items():                tgt_idx = self._order.index(tgt_lbl)                out[:, tgt_idx] = base[:, src_lbl]            return out        def __getattr__(self, name):            return getattr(self.model, name)    def patched_train_model(X_train, y_train, config):        if config.model_name.lower() == "xgb":            label_map = {-1: 0, 0: 1, 1: 2}            inv_map = {v: k for k, v in label_map.items()}            y_shift = np.array([label_map[int(v)] for v in y_train])            base = _train_xgb(X_train, y_shift, **config.params)            return XGBLabelWrapper(base, inv_map)        return _orig_train_model(X_train, y_train, config)    ml.train_model = patched_train_model    xgb_cfg = ContractRunConfig(        candidate_C=[0.1],        model_name="xgb",        param_grid=[{"n_estimators": 120, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.8, "colsample_bytree": 0.8, "n_jobs": -1}],        train_months=12,        test_months=1,        downsample_every=5,    )    xgb_results = {}    for code, path in contract_paths.items():        tag = short_contract_tag(path.stem)        res = run_single_contract(path, OUTPUT_ROOT, xgb_cfg, load_cfg)        xgb_results[code] = res        print(f"[XGB][{code}] Best params: {res.get('best_params')}")        print(f"[XGB][{code}] Summary head:{res.get('summary').head()}")

In [ ]:
if xgb_available and xgb_results is not None:    for code, res in xgb_results.items():        plot_confusion_and_rolling("xgb", short_contract_tag(res.get("contract", code)), res)else:    print("[XGB] Visualization skipped because training did not run.")